<a href="https://colab.research.google.com/github/Banty825413/Ml-and-DL-projects-/blob/main/.OptimizeCNNModl.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# New Section

In [1]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import mnist
import numpy as np

In [2]:
(train_image,train_label),(test_image,test_label) = mnist.load_data()

In [3]:
# Changing the data in 0 and 1
train_image = train_image/255.0
test_image = test_image/255.0

In [4]:
train_label[1]

np.uint8(0)

In [5]:
#Reshaping the data
train_image = train_image.reshape(len(train_image),28,28,1)

In [6]:
test_image = test_image.reshape(len(test_image),28,28,1)

In [7]:
# Defing the model
def build_model(hp):
  model = keras.Sequential(
      [
          keras.layers.Conv2D(
              filters = hp.Int('conv_1_filter',min_value =32,max_value = 128, step=16),
              kernel_size = hp.Choice('conv_1_kernel',values =[3,5]),
              activation = 'relu',
              input_shape = (28,28,1)

          ),
          keras.layers.Conv2D(
              filters = hp.Int('conv_2_filter',min_value = 32, max_value=128 , step = 16),
              kernel_size = hp.Choice('conv_2_kernel',values = [3,5]),
              activation ='relu'
          ),
          keras.layers.Flatten()
          ,
          keras.layers.Dense(
            units=hp.Int('dens_1_units',min_value= 32, max_value = 128, step =16),
            activation = 'relu'
           ),
          keras.layers.Dense(10,activation = 'softmax')
      ])
  # lets compile
  model.compile(optimizer = keras.optimizers.Adam(hp.Choice('learning_rate',values=[1e-2,1e-3])),
                loss  = 'sparse_categorical_crossentropy',
                metrics  = ['accuracy']

      )
  return model

In [8]:
import keras_tuner
from keras_tuner import RandomSearch
from keras_tuner.engine.hyperparameters import HyperParameters

In [9]:
tuner_search = RandomSearch(build_model,
                            objective='val_accuracy',
                            max_trials=3,
                            directory = 'output',
                            project_name = 'Mnist_fashion'

                            )

Reloading Tuner from output/Mnist_fashion/tuner0.json


In [10]:
tuner_search.search(train_image,train_label,epochs = 3,validation_split = 0.1)

In [11]:
tuner_search.results_summary()

Results summary
Results in output/Mnist_fashion
Showing 10 best trials
Objective(name="val_accuracy", direction="max")

Trial 1 summary
Hyperparameters:
conv_1_filter: 64
conv_1_kernel: 5
conv_2_filter: 128
conv_2_kernel: 5
dens_1_units: 80
learning_rate: 0.001
Score: 0.987333357334137

Trial 2 summary
Hyperparameters:
conv_1_filter: 48
conv_1_kernel: 3
conv_2_filter: 48
conv_2_kernel: 3
dens_1_units: 32
learning_rate: 0.001
Score: 0.9866666793823242

Trial 0 summary
Hyperparameters:
conv_1_filter: 48
conv_1_kernel: 3
conv_2_filter: 128
conv_2_kernel: 5
dens_1_units: 32
learning_rate: 0.01
Score: 0.9786666631698608


In [12]:
model= tuner_search.get_best_models(num_models=1)[0]

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 18 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [13]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 24, 24, 64)     │         1,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 20, 20, 128)    │       204,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 51200)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 80)             │     4,096,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │           810 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,303,482 (16.42 MB)

 Trainable params: 4,303,482 (16.42 MB)

 Non-trainable params: 0 (0.00 B)

In [14]:
model.fit(train_image,train_label,epochs=10,validation_split=0.1)

Epoch 1/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 19s 9ms/step - accuracy: 0.9921 - loss: 0.0254 - val_accuracy: 0.9923 - val_loss: 0.0294
Epoch 2/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - accuracy: 0.9956 - loss: 0.0131 - val_accuracy: 0.9893 - val_loss: 0.0522
Epoch 3/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - accuracy: 0.9962 - loss: 0.0119 - val_accuracy: 0.9900 - val_loss: 0.0468
Epoch 4/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 11s 6ms/step - accuracy: 0.9973 - loss: 0.0092 - val_accuracy: 0.9892 - val_loss: 0.0696
Epoch 5/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 11s 6ms/step - accuracy: 0.9971 - loss: 0.0099 - val_accuracy: 0.9900 - val_loss: 0.0501
Epoch 6/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 20s 6ms/step - accuracy: 0.9983 - loss: 0.0064 - val_accuracy: 0.9897 - val_loss: 0.0602
Epoch 7/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 20s 6ms/step - accuracy: 0.9976 - loss: 0.0068 - val_accuracy: 0.9908 - val_loss: 0.0688
Epoch 8/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - accuracy: 0.9982 - loss: 0

In [15]:
predict = model.predict(test_image)

313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step


array([[5.7444608e-18, 3.8675621e-20, 4.6645009e-17, ..., 1.0000000e+00,
        6.7508735e-11, 9.1101739e-09],
       [2.0301141e-22, 6.1593723e-36, 1.0000000e+00, ..., 1.4454069e-34,
        7.0093805e-32, 1.1109977e-31],
       [3.6442194e-19, 1.0000000e+00, 2.7108425e-18, ..., 5.6907963e-14,
        7.2995936e-17, 2.8269686e-20],
       ...,
       [1.7248741e-31, 1.6972585e-23, 2.3528869e-25, ..., 1.5967883e-16,
        8.6320162e-10, 1.8705606e-18],
       [1.4011072e-27, 1.3283112e-31, 1.1983904e-41, ..., 3.5876900e-36,
        7.1373601e-16, 6.4906642e-23],
       [1.6003412e-26, 9.2820140e-27, 3.7975908e-28, ..., 0.0000000e+00,
        4.8656066e-30, 2.2692643e-27]], dtype=float32)